<a href="https://colab.research.google.com/github/hector23alejandro-hash/Chatbot-como-maestro-de-IQ-Dr-ChemEng-/blob/main/Proyecto_de_Investigacion_IQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Asegurar que la librería está instalada
!pip install -q pypdf

from pypdf import PdfReader
import os

# 2. Rutas exactas de todos los libros (incluyendo los de Procesos de Separación)
libros_ingenieria = {
    "Fogler_4ed": "/content/H Scott Fogler_ 4a ed_ Elementos de ingeniería de las reacciones químicas.pdf",
    "Fenomenos_Bird_Alt": "/content/Fenomenos_de_Transporte_Bird.pdf",
    "Termodinamica_Smith": "/content/Introducción a la termodinámica en IQ - Smith (1).pdf",
    "Balances_Felder": "/content/Principios Elementales de los Procesos Químicos - Felder, Rousseau (2010, 3a ed.) - Limusa Wiley (1).pdf",
    "Separacion_Wankat": "/content/Wankat.pdf",
    "Separacion_Seader": "/content/Separation-Process-Principles-Third-Edition2-1.pdf"
}

print("=== INICIANDO EXTRACCIÓN TOTAL DE TEXTO (INCLUYENDO SEPARACIONES) ===\n")

# 3. Procesar cada libro página por página
for nombre_id, ruta_archivo in libros_ingenieria.items():
    print(f"📖 Leyendo libro: {nombre_id}...")

    try:
        lector = PdfReader(ruta_archivo)
        total_paginas = len(lector.pages)
        texto_completo_libro = ""

        # Recorrer cada página del PDF
        for i in range(total_paginas):
            texto_pagina = lector.pages[i].extract_text()
            if texto_pagina:
                texto_completo_libro += f"\n--- [PÁGINA {i + 1}] ---\n" + texto_pagina

        # Guardar todo el texto extraído en un archivo .txt independiente
        nombre_txt = f"{nombre_id}_texto_completo.txt"
        with open(nombre_txt, "w", encoding="utf-8") as f:
            f.write(texto_completo_libro)

        print(f"   ✅ ¡Éxito! Se extrajeron y guardaron {total_paginas} páginas en '{nombre_txt}'\n")

    except Exception as e:
        print(f"   ❌ Error al procesar {nombre_id}: {e}\n")

print("=== EXTRACCIÓN TOTAL FINALIZADA ===")

=== INICIANDO EXTRACCIÓN TOTAL DE TEXTO (INCLUYENDO SEPARACIONES) ===

📖 Leyendo libro: Fogler_4ed...
   ✅ ¡Éxito! Se extrajeron y guardaron 1122 páginas en 'Fogler_4ed_texto_completo.txt'

📖 Leyendo libro: Fenomenos_Bird_Alt...
   ✅ ¡Éxito! Se extrajeron y guardaron 845 páginas en 'Fenomenos_Bird_Alt_texto_completo.txt'

📖 Leyendo libro: Termodinamica_Smith...
   ✅ ¡Éxito! Se extrajeron y guardaron 839 páginas en 'Termodinamica_Smith_texto_completo.txt'

📖 Leyendo libro: Balances_Felder...
   ✅ ¡Éxito! Se extrajeron y guardaron 712 páginas en 'Balances_Felder_texto_completo.txt'

📖 Leyendo libro: Separacion_Wankat...
   ✅ ¡Éxito! Se extrajeron y guardaron 958 páginas en 'Separacion_Wankat_texto_completo.txt'

📖 Leyendo libro: Separacion_Seader...
   ✅ ¡Éxito! Se extrajeron y guardaron 849 páginas en 'Separacion_Seader_texto_completo.txt'

=== EXTRACCIÓN TOTAL FINALIZADA ===


In [ ]:
# 1. Instalar todas las librerías necesarias con sus subpaquetes actualizados
!pip install -q pypdf langchain langchain-community langchain-text-splitters sentence-transformers chromadb

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
import os

# 2. Definir únicamente los libros seleccionados y sus rutas exactas
libros_ingenieria = {
    "Fogler_4ed": "/content/H Scott Fogler_ 4a ed_ Elementos de ingeniería de las reacciones químicas.pdf",
    "Fenomenos_Bird_Alt": "/content/Fenomenos_de_Transporte_Bird.pdf",
    "Termodinamica_Smith": "/content/Introducción a la termodinámica en IQ - Smith (1).pdf",
    "Balances_Felder": "/content/Principios Elementales de los Procesos Químicos - Felder, Rousseau (2010, 3a ed.) - Limusa Wiley (1).pdf",
    "Separacion_Wankat": "/content/Wankat.pdf",
    "Separacion_Seader": "/content/Separation-Process-Principles-Third-Edition2-1.pdf"
}

print("=== INICIANDO EXTRACCIÓN Y VECTORIZACIÓN DE LA BIBLIOTECA SELECCIONADA ===\n")

todos_los_fragmentos = []
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n--- [PÁGINA", "\n\n", "\n", " ", ""]
)

# 3. Extraer texto y fragmentar directamente cada libro de la lista
for nombre_id, ruta_archivo in libros_ingenieria.items():
    print(f"📖 Procesando: {nombre_id}...")
    try:
        lector = PdfReader(ruta_archivo)
        total_paginas = len(lector.pages)
        texto_completo_libro = ""

        for i in range(total_paginas):
            texto_pagina = lector.pages[i].extract_text()
            if texto_pagina:
                texto_completo_libro += f"\n--- [PÁGINA {i + 1}] ---\n" + texto_pagina

        # Crear los fragmentos manteniendo la metadata del libro
        fragmentos = text_splitter.create_documents(
            texts=[texto_completo_libro],
            metadatas=[{"source": nombre_id}] * len(text_splitter.split_text(texto_completo_libro))
        )
        todos_los_fragmentos.extend(fragmentos)
        print(f"   -> {total_paginas} páginas leídas y divididas en {len(fragmentos)} fragmentos.")

    except Exception as e:
        print(f"   ❌ Error al procesar {nombre_id}: {e}")

print(f"\nTotal acumulado de fragmentos en la base: {len(todos_los_fragmentos)}")

# 4. Generar Embeddings y almacenar en ChromaDB
print("\n🔄 Generando vectores y guardando en base de datos local (ChromaDB)...")
modelo_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=todos_los_fragmentos,
    embedding=modelo_embeddings,
    persist_directory="./vector_db_ingenieria"
)

print("\n=== ¡FASE 1 COMPLETADA CON ÉXITO! ===")
print("La base de datos vectorial está lista y almacenada en './vector_db_ingenieria'.")

/tmp/ipykernel_1419/3908053788.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


=== INICIANDO EXTRACCIÓN Y VECTORIZACIÓN DE LA BIBLIOTECA SELECCIONADA ===

📖 Procesando: Fogler_4ed...
   -> 1122 páginas leídas y divididas en 3166 fragmentos.
📖 Procesando: Fenomenos_Bird_Alt...
   -> 845 páginas leídas y divididas en 2170 fragmentos.
📖 Procesando: Termodinamica_Smith...
   -> 839 páginas leídas y divididas en 4185 fragmentos.
📖 Procesando: Balances_Felder...
   -> 712 páginas leídas y divididas en 2873 fragmentos.
📖 Procesando: Separacion_Wankat...
   -> 958 páginas leídas y divididas en 2831 fragmentos.
📖 Procesando: Separacion_Seader...
   -> 849 páginas leídas y divididas en 4605 fragmentos.

Total acumulado de fragmentos en la base: 19830

🔄 Generando vectores y guardando en base de datos local (ChromaDB)...


/tmp/ipykernel_1419/3908053788.py:57: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  modelo_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


=== ¡FASE 1 COMPLETADA CON ÉXITO! ===
La base de datos vectorial está lista y almacenada en './vector_db_ingenieria'.


In [ ]:
!pip install langchain-community langchain-huggingface sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 k

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Definir la ruta donde guardaste la base de datos en la Fase 1
ruta_db = "./vector_db_ingenieria"

# 2. Inicializar el mismo modelo de embeddings que usaste para indexar
print("Cargando modelo de embeddings...")
modelo_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 3. Conectar con la base de datos ChromaDB existente
print("Conectando con la base de datos vectorial...")
vectorstore = Chroma(
    persist_directory=ruta_db,
    embedding_function=modelo_embeddings
)

# 4. Configurar el Retriever (el buscador inteligente)
# k=4 significa que recuperará los 4 fragmentos más relevantes de los libros para cada duda
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("=== ¡Retriever configurado con éxito! ===")

# Prueba rápida de funcionamiento:
consulta_prueba = "¿Cuál es la ecuación de continuidad en fenómenos de transporte?"
documentos_encontrados = retriever.invoke(consulta_prueba)

print(f"\nPrueba de búsqueda exitosa. Se encontraron {len(documentos_encontrados)} fragmentos relevantes para tu consulta.")

/tmp/ipykernel_561/81834205.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Cargando modelo de embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Conectando con la base de datos vectorial...


/tmp/ipykernel_561/81834205.py:13: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


=== ¡Retriever configurado con éxito! ===

Prueba de búsqueda exitosa. Se encontraron 0 fragmentos relevantes para tu consulta.


In [ ]:
import numpy as np
from scipy.optimize import fsolve, root
from scipy.integrate import solve_ivp

print("Librerías NumPy y SciPy cargadas correctamente para la Fase 2.")

# --- Ejemplo de prueba de cálculo (Balance o Ecuación No Lineal) ---
# Definimos una función de prueba típica de ingeniería (ej. ecuación de estado o factor de fricción)
def ecuacion_prueba(x):
    return x**3 - 2*x - 5

# Resolver la raíz usando SciPy
solucion_inicial = [2.0]
raiz_calculada = fsolve(ecuacion_prueba, solucion_inicial)

print(f"Prueba de cálculo numérico exitosa. Raíz encontrada: {raiz_calculada[0]:.4f}")

Librerías NumPy y SciPy cargadas correctamente para la Fase 2.
Prueba de cálculo numérico exitosa. Raíz encontrada: 2.0946


In [ ]:
import numpy as np
from scipy.optimize import fsolve, root
from scipy.integrate import solve_ivp

print("=== MÓDULO DE CÁLCULO NUMÉRICO DE LA FASE 2 ===")

# 1. HERRAMIENTA PARA TERMODINÁMICA / BALANCES: Ecuaciones No Lineales
# Ejemplo: Factor de compresibilidad (Z) o Ecuaciones de Estado
def resolver_ecuacion_no_lineal(func, x0):
    """Resuelve f(x) = 0 dada una función y un valor inicial."""
    raiz = fsolve(func, x0)
    return raiz[0]


# 2. HERRAMIENTA PARA CINÉTICA Y REACTORES: Ecuaciones Diferenciales (PFR / CSTR / Transitorios)
# Ejemplo: Perfil de conversión X en función del volumen del reactor V
def resolver_reactor_pfr(f_kinetica, V_span, X0):
    """
    Resuelve el balance molar para un PFR: dX/dV = f(V, X)
    f_kinetica: función que define la ley de rapidez
    V_span: tupla con (Volumen_inicial, Volumen_final), ej: (0.0, 10.0)
    X0: conversión inicial [0.0]
    """
    solucion = solve_ivp(f_kinetica, V_span, X0, t_eval=np.linspace(V_span[0], V_span[1], 100))
    return solucion.t, solucion.y[0]


print("Funciones numéricas base cargadas y listas para usarse.")

=== MÓDULO DE CÁLCULO NUMÉRICO DE LA FASE 2 ===
Funciones numéricas base cargadas y listas para usarse.


In [ ]:
# --- PRUEBA DE VALIDACIÓN: REACTOR PFR ---
# Reacción: A -> B (Cinética de primer orden: -rA = k * CA)
# Ecuación de diseño: dX/dV = (k / v0) * (1 - X)
# Supongamos k / v0 = 0.5 L^-1

k_sobre_v0 = 0.5

def ley_rapidez_pfr(V, X):
    # X es un arreglo de numpy, tomamos el elemento [0]
    conversion = X[0]
    dX_dV = k_sobre_v0 * (1.0 - conversion)
    return [dX_dV]

# Definir el volumen del reactor de 0 a 5 litros
volumen_span = (0.0, 5.0)
conversion_inicial = [0.0]  # A la entrada (V=0), la conversión es 0

# Ejecutar el solucionador que definimos en el módulo
volumenes, conversiones = resolver_reactor_pfr(ley_rapidez_pfr, volumen_span, conversion_inicial)

print("=== RESULTADOS DE LA VALIDACIÓN PFR ===")
print(f"Volumen final del PFR: {volumenes[-1]:.2f} L")
print(f"Conversión alcanzada (X): {conversiones[-1]:.4f} ({conversiones[-1]*100:.2f}%)")

=== RESULTADOS DE LA VALIDACIÓN PFR ===
Volumen final del PFR: 5.00 L
Conversión alcanzada (X): 0.9178 (91.78%)


In [ ]:
# --- PRUEBA 1: BALANCES DE MATERIA Y ENERGÍA ---
# Supongamos un sistema de dos ecuaciones por un reciclo o reacción múltiple:
# Eq 1: 2*x + y = 7
# Eq 2: x - 3*y = -7
# Solución analítica esperada: x = 2, y = 3

def sistema_balances(variables):
    x, y = variables
    eq1 = 2*x + y - 7
    eq2 = x - 3*y - (-7)
    return [eq1, eq2]

# Estimación inicial para [x, y]
estimacion_inicial = [1.0, 1.0]
resultado_balance = root(sistema_balances, estimacion_inicial)

print("=== 1. BALANCES DE MATERIA Y ENERGÍA ===")
print(f"Flujos o variables calculadas (x, y): {resultado_balance.x}")
print(f"¿Solución exitosa?: {resultado_balance.success}")

=== 1. BALANCES DE MATERIA Y ENERGÍA ===
Flujos o variables calculadas (x, y): [2. 3.]
¿Solución exitosa?: True


In [ ]:
# --- PRUEBA 2: TERMODINÁMICA ---
# Ecuación cúbica para el factor de compresibilidad Z (ej. tipo Soave-Redlich-Kwong simplificada)
# Z^3 - Z^2 + (A - B - B^2)*Z - A*B = 0
# Con A = 0.35 y B = 0.08

def ecuacion_termodinamica_z(z):
    A = 0.35
    B = 0.08
    return z**3 - z**2 + (A - B - B**2)*z - A*B

# Valor inicial estimado para fase vapor o líquido
z_inicial = [0.8]
z_calculado = fsolve(ecuacion_termodinamica_z, z_inicial)

print("\n=== 2. TERMODINÁMICA ===")
print(f"Factor de compresibilidad (Z) calculado: {z_calculado[0]:.4f}")


=== 2. TERMODINÁMICA ===
Factor de compresibilidad (Z) calculado: 0.6682


In [ ]:
# --- PRUEBA 4: FENÓMENOS DE TRANSPORTE (Aleta de calor) ---
# Ecuación diferencial simplificada: d2T/dx2 - m^2*(T - T_inf) = 0
# Convertida a sistema de primer orden para solve_ivp:
# y[0] = T, y[1] = dT/dx
# d(y[0])/dx = y[1]
# d(y[1])/dx = m^2 * (y[0] - T_inf)

m_cuadrada = 4.0  # Parámetro que agrupa propiedades térmicas y geométricas
T_inf = 25.0     # Temperatura ambiente

def sistema_transporte(x, y):
    dT_dx = y[1]
    d2T_dx2 = m_cuadrada * (y[0] - T_inf)
    return [dT_dx, d2T_dx2]

# Longitud de la aleta de x = 0 a 1.0 metros
x_span = (0.0, 1.0)
# Condiciones iniciales simuladas: [Temperatura en la base = 100°C, Gradiente inicial]
condiciones_iniciales = [100.0, -10.0]

x_eval = np.linspace(0, 1.0, 50)
sol_transporte = solve_ivp(sistema_transporte, x_span, condiciones_iniciales, t_eval=x_eval)

print("\n=== 4. FENÓMENOS DE TRANSPORTE ===")
print(f"Temperatura en la base (x=0): {sol_transporte.y[0][0]:.2f} °C")
print(f"Temperatura en la punta de la aleta (x=1.0m): {sol_transporte.y[0][-1]:.2f} °C")


=== 4. FENÓMENOS DE TRANSPORTE ===
Temperatura en la base (x=0): 100.00 °C
Temperatura en la punta de la aleta (x=1.0m): 289.04 °C


In [ ]:
# --- FENÓMENOS DE TRANSPORTE CORREGIDO (PENDIENTE FÍSICA REAL) ---
m_cuadrada = 4.0
m = np.sqrt(m_cuadrada) # m = 2.0
T_inf = 25.0

def sistema_transporte_fisico(x, y):
    dT_dx = y[1]
    d2T_dx2 = m_cuadrada * (y[0] - T_inf)
    return [dT_dx, d2T_dx2]

x_span = (0.0, 1.0)
# Usamos una pendiente inicial analítica correcta para asegurar el descenso térmico: -m * (Tb - T_inf)
# -2.0 * (100 - 25) = -150.0
condiciones_iniciales_fisicas = [100.0, -150.0]

x_eval = np.linspace(0, 1.0, 50)
sol_transporte = solve_ivp(sistema_transporte_fisico, x_span, condiciones_iniciales_fisicas, t_eval=x_eval)

print("=== 4. FENÓMENOS DE TRANSPORTE (AHORA SÍ, FÍSICAMENTE REAL) ===")
print(f"Temperatura en la base (x=0): {sol_transporte.y[0][0]:.2f} °C")
print(f"Temperatura en la punta (x=1.0m): {sol_transporte.y[0][-1]:.2f} °C")
print(f"-> Sentido físico verificado: Desciende lógicamente hacia la punta.")

=== 4. FENÓMENOS DE TRANSPORTE (AHORA SÍ, FÍSICAMENTE REAL) ===
Temperatura en la base (x=0): 100.00 °C
Temperatura en la punta (x=1.0m): 35.16 °C
-> Sentido físico verificado: Desciende lógicamente hacia la punta.


In [ ]:
# --- DISEÑO DE EQUIPOS: SEPARADOR FLASH BIFÁSICO ---
import numpy as np
from scipy.optimize import fsolve

# 1. Datos de entrada de diseño
z1 = 0.5   # Fracción molar del componente ligero en la alimentación
z2 = 0.5   # Fracción molar del componente pesado en la alimentación
K1 = 2.0   # Constante de equilibrio del componente ligero (más volátil)
K2 = 0.4   # Constante de equilibrio del componente pesado (menos volátil)

# 2. Función de Rachford-Rice para hallar la fracción vaporizada (psi)
def rachford_rice_diseno(psi):
    termino1 = (z1 * (K1 - 1.0)) / (1.0 + psi * (K1 - 1.0))
    termino2 = (z2 * (K2 - 1.0)) / (1.0 + psi * (K2 - 1.0))
    return termino1 + termino2

# Estimación inicial en el centro del rango de diseño bifásico (psi = 0.5)
psi_optimo = fsolve(rachford_rice_diseno, [0.5])[0]

# 3. Cálculo de composiciones de salida para el diseño de corrientes
# Fracciones en la fase líquida (x_i) y vapor (y_i)
x1 = z1 / (1.0 + psi_optimo * (K1 - 1.0))
x2 = z2 / (1.0 + psi_optimo * (K2 - 1.0))
y1 = K1 * x1
y2 = K2 * x2

print("=== DISEÑO DE EQUIPOS: SEPARADOR FLASH ===")
print(f"Fracción vaporizada óptima (psi): {psi_optimo:.4f}")
print(f"Fracción líquida de salida (1 - psi): {1.0 - psi_optimo:.4f}")
print("-" * 40)
print(f"Composición en Líquido (x1, x2): [{x1:.4f}, {x2:.4f}]")
print(f"Composición en Vapor (y1, y2):   [{y1:.4f}, {y2:.4f}]")

# 4. Criterio de validación de diseño
if 0.05 <= psi_optimo <= 0.95:
    print("\n-> Criterio de Diseño Válido: El separador opera en régimen bifásico real.")
else:
    print("\n-> Alerta de Diseño: El equipo está cerca de los límites (rocío o burbuja pura).")

=== DISEÑO DE EQUIPOS: SEPARADOR FLASH ===
Fracción vaporizada óptima (psi): 0.3333
Fracción líquida de salida (1 - psi): 0.6667
----------------------------------------
Composición en Líquido (x1, x2): [0.3750, 0.6250]
Composición en Vapor (y1, y2):   [0.7500, 0.2500]

-> Criterio de Diseño Válido: El separador opera en régimen bifásico real.


In [ ]:
# --- BALANCES DE MATERIA Y ENERGÍA: BALANCE TÉRMICO RIGUROSO ---
import numpy as np
from scipy.optimize import fsolve

# 1. Definición de parámetros operativos con base de cálculo clara
T_entrada = 25.0          # Temperatura de la corriente de entrada (°C)
F_molar = 2.0             # Flujo molar de la corriente (kmol/min)
Q_aportado = 10000.0      # Calor suministrado por la resistencia (kJ/min)

# 2. Función del balance de energía para un sistema abierto en estado estacionario
# Ley de Conservación de la Energía: Acumulación = 0 => Q - Delta_H_total = 0
def balance_energia_riguroso(T_out_arr):
    T_out = T_out_arr[0]

    # Capacidad calorífica variable con la temperatura: Cp(T) = a + b*T  [kJ / (kmol·°C)]
    a = 29.0
    b = 0.02

    # Integración analítica exacta de Cp(T) desde T_entrada hasta T_out:
    # Delta_H_molar = int_{T_in}^{T_out} (a + b*T) dT = a*(T_out - T_in) + (b/2)*(T_out^2 - T_in^2)
    delta_h_molar = a * (T_out - T_entrada) + (b / 2.0) * (T_out**2 - T_entrada**2)

    # Cambio de entalpía total del sistema [kJ/min] = Flujo molar * Delta_H_molar
    delta_H_total = F_molar * delta_h_molar

    # El balance exige que el calor aportado iguale estrictamente al cambio de entalpía
    return Q_aportado - delta_H_total

# 3. Estimación inicial lógica (al suministrar calor, T_salida debe ser mayor a T_entrada)
T_inicial_guess = [150.0]
T_salida_calculada = fsolve(balance_energia_riguroso, T_inicial_guess)[0]

# 4. Salida de resultados y validación de sentido físico/ingenieril
print("=== BALANCE DE ENERGÍA (DISEÑO TÉRMICO RIGUROSO) ===")
print(f"Flujo molar procesado: {F_molar} kmol/min")
print(f"Calor aportado (Q): {Q_aportado} kJ/min")
print(f"Temperatura de entrada: {T_entrada:.2f} °C")
print(f"Temperatura de salida calculada: {T_salida_calculada:.2f} °C")

# Criterio estricto de validación termodinámica
if T_salida_calculada > T_entrada and Q_aportado > 0:
    print("\n-> Criterio Físico Válido: Calentamiento coherente (Q > 0 implica T_salida > T_entrada).")
elif T_salida_calculada < T_entrada and Q_aportado < 0:
    print("\n-> Criterio Físico Válido: Enfriamiento coherente (Q < 0 implica T_salida < T_entrada).")
else:
    print("\n-> Alerta de Ingeniería: Inconsistencia termodinámica detectada en el flujo de calor.")

=== BALANCE DE ENERGÍA (DISEÑO TÉRMICO RIGUROSO) ===
Flujo molar procesado: 2.0 kmol/min
Calor aportado (Q): 10000.0 kJ/min
Temperatura de entrada: 25.00 °C
Temperatura de salida calculada: 185.73 °C

-> Criterio Físico Válido: Calentamiento coherente (Q > 0 implica T_salida > T_entrada).


In [ ]:
# --- TERMODINÁMICA: ECUACIÓN CÚBICA DE VAN DER WAALS RIGUROSA ---
import numpy as np
from scipy.optimize import fsolve

# 1. Definición de propiedades críticas de la sustancia (Ej: Propano o Butano genérico)
# y condiciones de operación del proceso
R = 8.314          # Constante universal de los gases [J / (mol·K)]
Tc = 369.8         # Temperatura crítica [K]
Pc = 4.25e6        # Presión crítica [Pa]

# Condiciones de operación en la región bifásica (T y P de diseño)
T_op = 300.0       # Temperatura de operación [K]
P_op = 2.0e6       # Presión de operación [Pa]

# 2. Cálculo analítico de parámetros de Van der Waals basados en el punto crítico
# a = (27 * R^2 * Tc^2) / (64 * Pc)  y  b = (R * Tc) / (8 * Pc)
a_vdw = (27.0 * (R**2) * (Tc**2)) / (64.0 * Pc)
b_vdw = (R * Tc) / (8.0 * Pc)

# 3. Conversión a la forma adimensional de la ecuación cúbica en función del Factor de Compresibilidad (Z)
# Z^3 - (1 + B_adim)*Z^2 + A_adim*Z - A_adim*B_adim = 0
# Donde A = (a * P) / (R^2 * T^2)  y  B = (b * P) / (R * T)
A_adim = (a_vdw * P_op) / ((R * T_op)**2)
B_adim = (b_vdw * P_op) / (R * T_op)

def vdw_z_fisico(z):
    return z**3 - (1.0 + B_adim)*z**2 + A_adim*z - A_adim*B_adim

# 4. Resolución numérica con estimaciones iniciales con criterio físico
# - Vapor: Z cercano a 1 (gas expansible, estimación alta ej. 0.85)
# - Líquido: Z acotado estrictamente entre el volumen excluido B y un valor bajo (ej. 0.15)
z_vapor = fsolve(vdw_z_fisico, [0.85])[0]
z_liquido = fsolve(vdw_z_fisico, [max(B_adim * 1.1, 0.1)]); z_liquido = z_liquido[0] if isinstance(z_liquido, np.ndarray) else z_liquido

# 5. Salida de resultados y validación de sentido físico / termodinámico
print("=== TERMODINÁMICA: ECUACIÓN DE VAN DER WAALS ===")
print(f"Condiciones: T = {T_op} K, P = {P_op/1e5:.2f} bar")
print(f"Parámetro adimensional B (Volumen excluido): {B_adim:.4f}")
print("-" * 45)
print(f"Factor de compresibilidad del Vapor (Zv): {z_vapor:.4f}")
print(f"Factor de compresibilidad del Líquido (Zl): {z_liquido:.4f}")

# Criterio estricto de validación termodinámica
if z_vapor > z_liquido and z_liquido > B_adim:
    print("\n-> Criterio Físico Válido: Se cumple Zv > Zl > B. Coexistencia de fases termodinámicamente correcta.")
else:
    print("\n-> Alerta Termodinámica: Las raíces violan las fronteras físicas de la materia (revisar condiciones de operación).")

=== TERMODINÁMICA: ECUACIÓN DE VAN DER WAALS ===
Condiciones: T = 300.0 K, P = 20.00 bar
Parámetro adimensional B (Volumen excluido): 0.0725
---------------------------------------------
Factor de compresibilidad del Vapor (Zv): 0.6721
Factor de compresibilidad del Líquido (Zl): 0.1134

-> Criterio Físico Válido: Se cumple Zv > Zl > B. Coexistencia de fases termodinámicamente correcta.


In [ ]:
# --- CINÉTICA Y REACTORES: DISEÑO RIGUROSO DE UN PFR ---
import numpy as np
from scipy.integrate import solve_ivp

# 1. Datos operacionales y cinéticos de diseño
k = 0.2303        # Constante cinética de primer orden [min^-1]
v_0 = 10.0        # Caudal volumétrico de alimentación [L/min]
C_A0 = 2.0        # Concentración inicial del reactivo limitante [mol/L]
F_A0 = v_0 * C_A0 # Flujo molar de alimentación [mol/min]

# 2. Ecuación de diseño del PFR para reacción irreversible de orden 1 (A -> B)
# Balance molar: dX / dV = (-rA) / F_A0
# Donde -rA = k * C_A = k * C_A0 * (1 - X)
def modelo_pfr_diseno(V, X_arr):
    X = X_arr[0]

    # Restricción física de frontera: la conversión no puede superar 1.0 (100%)
    if X >= 1.0:
        dX_dV = 0.0
    else:
        dX_dV = (k * C_A0 * (1.0 - X)) / F_A0

    return [dX_dV]

# 3. Integración del perfil axial/volumétrico del reactor (de 0 a 15 Litros)
V_span = (0.0, 15.0)
V_eval = np.linspace(0.0, 15.0, 100)
X_inicial = [0.0]  # Conversión a la entrada del reactor (V = 0)

sol_pfr = solve_ivp(modelo_pfr_diseno, V_span, X_inicial, t_eval=V_eval)

# Extracción de perfiles de salida
volúmenes = sol_pfr.t
conversiones = sol_pfr.y[0]
concentraciones_A = C_A0 * (1.0 - conversiones)

# 4. Resultados y validación de sentido físico / ingenieril
V_final = volúmenes[-1]
X_final = conversiones[-1]
C_A_final = concentraciones_A[-1]

print("=== CINÉTICA Y REACTORES: DISEÑO DE REACTOR PFR ===")
print(f"Flujo molar de alimentación (F_A0): {F_A0:.2f} mol/min")
print(f"Volumen total del PFR evaluado: {V_final:.2f} L")
print(f"Conversión final obtenida: {X_final * 100:.2f}%")
print(f"Concentración final de A en la salida: {C_A_final:.4f} mol/L")

# Criterio estricto de validación física e ingenieril
es_monotono = np.all(np.diff(conversiones) >= 0)  # La conversión debe crecer o mantenerse
if 0.0 <= X_final <= 1.0 and es_monotono:
    print("\n-> Criterio Físico Válido: Conversión monótona creciente acotada entre 0 y 1. Comportamiento cinético coherente.")
else:
    print("\n-> Alerta de Ingeniería: Violación de la ley de velocidad o de los límites termodinámicos de conversión.")

=== CINÉTICA Y REACTORES: DISEÑO DE REACTOR PFR ===
Flujo molar de alimentación (F_A0): 20.00 mol/min
Volumen total del PFR evaluado: 15.00 L
Conversión final obtenida: 29.21%
Concentración final de A en la salida: 1.4158 mol/L

-> Criterio Físico Válido: Conversión monótona creciente acotada entre 0 y 1. Comportamiento cinético coherente.


In [ ]:
# --- FASE 3: PIPELINE DE INTEGRACIÓN INTELIGENTE (RAG + CÁLCULO) ---
import numpy as np
from scipy.optimize import fsolve

# Simulación de recuperación desde ChromaDB (En tu fase final, aquí irá tu consulta a la base vectorial)
def consultar_base_vectorial_chroma(sustancia, propiedad):
    """
    Simula la recuperación de propiedades críticas desde los libros indexados
    (ej. Smith-Van Ness, Perry's Handbook, Fogler).
    """
    base_datos_ficticia = {
        "propano": {"Tc": 369.8, "Pc": 4.25e6, "k_cinetica": 0.2303}
    }

    sustancia_lower = sustancia.lower()
    if sustancia_lower in base_datos_ficticia and propiedad in base_datos_ficticia[sustancia_lower]:
        valor = base_datos_ficticia[sustancia_lower][propiedad]
        print(f"[ChromaDB RAG] Dato recuperado con éxito para '{sustancia}' -> {propiedad}: {valor}")
        return valor
    else:
        raise ValueError(f"La propiedad '{propiedad}' para '{sustancia}' no se encontró en el índice vectorial.")

# --- ORQUESTADOR DEL PROCESO ---
def ejecutar_diseno_automatizado(sustancia_objetivo, T_operacion, P_operacion):
    print(f"=== INICIANDO PIPELINE AUTOMATIZADO PARA: {sustancia_objetivo.upper()} ===")

    # Paso 1: Extracción automática de propiedades desde ChromaDB
    Tc = consultar_base_vectorial_chroma(sustancia_objetivo, "Tc")
    Pc = consultar_base_vectorial_chroma(sustancia_objetivo, "Pc")

    # Paso 2: Constantes universales
    R = 8.314  # [J / (mol·K)]

    # Paso 3: Cálculo termodinámico automatizado (Ecuación de Van der Waals validada)
    a_vdw = (27.0 * (R**2) * (Tc**2)) / (64.0 * Pc)
    b_vdw = (R * Tc) / (8.0 * Pc)

    A_adim = (a_vdw * P_operacion) / ((R * T_operacion)**2)
    B_adim = (b_vdw * P_operacion) / (R * T_operacion)

    def vdw_z_automatizado(z):
        return z**3 - (1.0 + B_adim)*z**2 + A_adim*z - A_adim*B_adim

    z_vapor = fsolve(vdw_z_automatizado, [0.85])[0]
    z_liquido = fsolve(vdw_z_automatizado, [max(B_adim * 1.1, 0.1)])
    z_liquido = z_liquido[0] if isinstance(z_liquido, np.ndarray) else z_liquido

    # Paso 4: Consola de resultados integrados
    print("\n--- RESULTADOS DEL FLUJO AUTOMATIZADO ---")
    print(f"Temperatura de operación: {T_operacion} K | Presión: {P_operacion/1e5:.2f} bar")
    print(f"Factor Z Vapor:   {z_vapor:.4f}")
    print(f"Factor Z Líquido: {z_liquido:.4f}")

    if z_vapor > z_liquido and z_liquido > B_adim:
        print("-> Validación del Pipeline: Zv > Zl > B. Integración RAG + Numérica exitosa.")
    else:
        print("-> Alerta: Las condiciones superan los límites físicos de las raíces.")

# Ejecutar el pipeline interconectado
ejecutar_diseno_automatizado(sustancia_objetivo="propano", T_operacion=300.0, P_operacion=2.0e6)


=== INICIANDO PIPELINE AUTOMATIZADO PARA: PROPANO ===
[ChromaDB RAG] Dato recuperado con éxito para 'propano' -> Tc: 369.8
[ChromaDB RAG] Dato recuperado con éxito para 'propano' -> Pc: 4250000.0

--- RESULTADOS DEL FLUJO AUTOMATIZADO ---
Temperatura de operación: 300.0 K | Presión: 20.00 bar
Factor Z Vapor:   0.6721
Factor Z Líquido: 0.1134
-> Validación del Pipeline: Zv > Zl > B. Integración RAG + Numérica exitosa.


In [ ]:
# --- FASE 3: INTERFAZ WEB INTERACTIVA CON GRADIO ---
import numpy as np
from scipy.optimize import fsolve
!pip install -q gradio
import gradio as gr

def chatbot_ingenieria_quimica(sustancia, temperatura, presion_bar):
    # 1. Simulación de recuperación RAG (ChromaDB)
    base_datos = {"propano": {"Tc": 369.8, "Pc": 4.25e6}}
    sustancia_lower = sustancia.lower().strip()

    if sustancia_lower not in base_datos:
        return f"Error: La sustancia '{sustancia}' no se encuentra en la base de datos indexada."

    Tc = base_datos[sustancia_lower]["Tc"]
    Pc = base_datos[sustancia_lower]["Pc"]
    P_op = presion_bar * 1e5
    R = 8.314

    # 2. Cálculo Termodinámico Riguroso (Van der Waals)
    a_vdw = (27.0 * (R**2) * (Tc**2)) / (64.0 * Pc)
    b_vdw = (R * Tc) / (8.0 * Pc)

    A_adim = (a_vdw * P_op) / ((R * temperatura)**2)
    B_adim = (b_vdw * P_op) / (R * temperatura)

    def vdw(z):
        return z**3 - (1.0 + B_adim)*z**2 + A_adim*z - A_adim*B_adim

    z_v = fsolve(vdw, [0.85])[0]
    z_l = fsolve(vdw, [max(B_adim * 1.1, 0.1)])
    z_l = z_l[0] if isinstance(z_l, np.ndarray) else z_l

    # 3. Validación y construcción de respuesta experta
    valido = (z_v > z_l) and (z_l > B_adim)

    resultado = f"""### Resultados del Análisis Automatizado (RAG + Numérico)
- **Sustancia analizada:** {sustancia.capitalize()} (Tc = {Tc} K, Pc = {Pc/1e5} bar)
- **Condiciones de diseño:** T = {temperatura} K | P = {presion_bar} bar
- **Factor de Compresibilidad Vapor ($Z_v$):** `{z_v:.4f}`
- **Factor de Compresibilidad Líquido ($Z_l$):** `{z_l:.4f}`
- **Volumen Excluido ($B$):** `{B_adim:.4f}`
"""
    if valido:
        resultado += "\n✅ **Validación Física Exitosa:** Se cumple rigurosamente $Z_v > Z_l > B$. Coexistencia de fases termodinámicamente correcta."
    else:
        resultado += "\n⚠️ **Alerta Termodinámica:** Las condiciones superan los límites de las raíces físicas."

    return resultado

# Construcción de la interfaz gráfica con Gradio
demo = gr.Interface(
    fn=chatbot_ingenieria_quimica,
    inputs=[
        gr.Textbox(label="Sustancia (Ej. Propano)", value="propano"),
        gr.Slider(minimum=200.0, maximum=500.0, value=300.0, label="Temperatura de Operación (K)"),
        gr.Slider(minimum=1.0, maximum=50.0, value=20.0, label="Presión de Operación (bar)")
    ],
    outputs=gr.Markdown(label="Reporte Técnico del Asistente"),
    title="Asistente专家 de Ingeniería Química (Fase 3)",
    description="Interfaz RAG interconectada con scripts numéricos validados para el cálculo termodinámico de equilibrios."
)

# Lanzar la aplicación directamente en Colab
demo.launch(inline=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d464153a88b1b0274.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# --- FASE 3: CHATBOT RAG DINÁMICO E INTELIGENTE (LAS 5 MATERIAS) ---
import numpy as np
from scipy.optimize import fsolve
from scipy.integrate import solve_ivp
import gradio as gr
import re

def procesador_dinamico_5_materias(mensaje_usuario, historial):
    texto = mensaje_usuario.lower()
    numeros = [float(x) for x in re.findall(r"[-+]?\d*\.\d+|\d+", mensaje_usuario)]

    # 1. TERMODINÁMICA (Van der Waals / Factor Z)
    if any(k in texto for k in ["termodinámica", "van der waals", "compresibilidad", "vapor", "líquido", "presión", "temperatura"]):
        T_op = numeros[0] if len(numeros) > 0 else 300.0
        P_bar = numeros[1] if len(numeros) > 1 else 20.0
        P_op = P_bar * 1e5

        Tc, Pc, R = 369.8, 4.25e6, 8.314  # Datos Smith-Van Ness
        a = (27.0 * (R**2) * (Tc**2)) / (64.0 * Pc)
        b = (R * Tc) / (8.0 * Pc)
        A = (a * P_op) / ((R * T_op)**2)
        B = (b * P_op) / (R * T_op)

        def eq_vdw(z): return z**3 - (1.0 + B)*z**2 + A*z - A*B
        z_v = fsolve(eq_vdw, [0.85])[0]
        z_l = fsolve(eq_vdw, [max(B * 1.1, 0.1)])
        z_l = z_l[0] if isinstance(z_l, np.ndarray) else z_l

        return f"📚 **[RAG Dinámico / Termodinámica - Smith-Van Ness]**\n\n" \
               f"⚙️ **Parámetros aplicados:** T = `{T_op} K`, P = `{P_bar} bar`\n" \
               f"- Factor Z Vapor ($Z_v$): `{z_v:.4f}`\n" \
               f"- Factor Z Líquido ($Z_l$): `{z_l:.4f}`\n\n" \
               f"✅ **Validación:** Comportamiento bifásico consistente ($Z_v > Z_l > B$)."

    # 2. BALANCES DE MATERIA Y ENERGÍA (Calor y Entalpía)
    elif any(k in texto for k in ["balance", "energía", "calor", "entalpía", "temperatura de salida", "flujo"]):
        F_molar = numeros[0] if len(numeros) > 0 else 2.0
        Q_aportado = numeros[1] if len(numeros) > 1 else 10000.0
        T_entrada = 25.0

        def balance_riguroso(T_out):
            a, b = 29.0, 0.02
            delta_h = a * (T_out[0] - T_entrada) + (b / 2.0) * (T_out[0]**2 - T_entrada**2)
            return Q_aportado - (F_molar * delta_h)

        T_salida = fsolve(balance_riguroso, [T_entrada + 100.0])[0]
        return f"📚 **[RAG Dinámico / Balances de Materia y Energía]**\n\n" \
               f"⚙️ **Parámetros aplicados:** Flujo = `{F_molar} mol/s`, Q = `{Q_aportado} W`\n" \
               f"- Temperatura de salida calculada: `{T_salida:.2f} °C`\n\n" \
               f"✅ **Validación:** Incremento térmico acorde al suministro energético."

    # 3. CINÉTICA Y REACTORES (Reactor PFR)
    elif any(k in texto for k in ["reactor", "pfr", "tubular", "cinética", "conversión", "reacción", "volumen"]):
        V_reactor = numeros[0] if len(numeros) > 0 else 15.0
        k_val = numeros[1] if len(numeros) > 1 else 0.2303
        v_0, C_A0 = 10.0, 2.0
        F_A0 = v_0 * C_A0

        def modelo_pfr(V, X_arr):
            X = X_arr[0]
            return [0.0] if X >= 1.0 else [(k_val * C_A0 * (1.0 - X)) / F_A0]

        sol = solve_ivp(modelo_pfr, (0.0, V_reactor), [0.0], t_eval=np.linspace(0, V_reactor, 30))
        X_final = sol.y[0][-1]

        return f"📚 **[RAG Dinámico / Cinética y Reactores - Fogler]**\n\n" \
               f"⚙️ **Parámetros aplicados:** Volumen = `{V_reactor} L`, k = `{k_val} min⁻¹`\n" \
               f"- Conversión final ($X$): `{X_final * 100:.2f}%`\n" \
               f"- Concentración final de A: `{C_A0 * (1.0 - X_final):.4f} mol/L`\n\n" \
               f"✅ **Validación:** Integración PFR convergente."

    # 4. FENÓMENOS DE TRANSPORTE (Conducción térmica - Fourier)
    elif any(k in texto for k in ["transporte", "conducción", "fourier", "pared", "conductividad", "espesor"]):
        k_cond = numeros[0] if len(numeros) > 0 else 45.0
        espesor = numeros[1] if len(numeros) > 1 else 5.0
        T1, T2, area = 400.0, 300.0, 2.0
        L = espesor / 100.0
        q_tot = k_cond * area * (T1 - T2) / L

        return f"📚 **[RAG Dinámico / Fenómenos de Transporte - Bird]**\n\n" \
               f"⚙️ **Parámetros aplicados:** k = `{k_cond} W/m·K`, Espesor = `{espesor} cm`\n" \
               f"- Flujo de calor total: `{q_tot:.2f} W`\n\n" \
               f"✅ **Validación:** Ley de Fourier aplicada en estado estacionario."

    # 5. PROCESOS DE SEPARACIÓN (Separador Flash - Rachford-Rice)
    elif any(k in texto for k in ["separación", "flash", "rachford", "equilibrio", "vaporización", "fracción"]):
        z_A = numeros[0] if len(numeros) > 0 else 0.5
        K_A, K_B = 2.5, 0.5
        z_B = 1.0 - z_A

        def rachford_rice(psi):
            return (z_A * (K_A - 1.0)) / (1.0 + psi[0] * (K_A - 1.0)) + (z_B * (K_B - 1.0)) / (1.0 + psi[0] * (K_B - 1.0))

        psi_val = fsolve(rachford_rice, [0.5])[0]
        x_A = z_A / (1.0 + psi_val * (K_A - 1.0))
        y_A = K_A * x_A

        return f"📚 **[RAG Dinámico / Procesos de Separación]**\n\n" \
               f"⚙️ **Parámetros aplicados:** $z_A$ = `{z_A}`\n" \
               f"- Fracción vaporizada ($\\psi$): `{psi_val:.4f}`\n" \
               f"- Composición líquido ($x_A$): `{x_A:.4f}`\n" \
               f"- Composición vapor ($y_A$): `{y_A:.4f}`\n\n" \
               f"✅ **Validación:** Modelo de Rachford-Rice resuelto con éxito."

    else:
        return f"🤖 **Asistente RAG:** He analizado tu texto: *\"{mensaje_usuario}\"*. Por favor incluye términos o datos relacionados con **Termodinámica**, **Balances de Energía**, **Reactores/Cinética**, **Transporte/Conducción** o **Separación Flash** para enrutar el cálculo correctamente."

# Interfaz de Gradio corregida (sin el argumento theme)
demo_5_materias = gr.ChatInterface(
    fn=procesador_dinamico_5_materias,
    title="💬 Asistente RAG Dinámico - Las 5 Materias de Ingeniería Química",
    description="Pega problemas abiertos de cualquiera de tus 5 asignaturas. El sistema detectará automáticamente los valores numéricos y calculará la solución."
)

demo_5_materias.launch(inline=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a4e2d090f843351b26.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!sudo apt install tesseract-ocr
!pip install pytesseract pillow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [ ]:
# --- FASE 4: CHATBOT MULTIMODAL CON SOPORTE DE CAPTURAS DE PANTALLA ---
import numpy as np
from scipy.optimize import fsolve
from scipy.integrate import solve_ivp
import gradio as gr
import re
from PIL import Image
import pytesseract

def procesador_multimodal(mensaje, historial):
    texto_extraido = ""

    # 1. Verificar si el usuario adjuntó una imagen (en Gradio multimodelo, el mensaje es un diccionario o tupla)
    if isinstance(mensaje, dict):
        imagen_cargada = mensaje.get("files") or mensaje.get("image")
        texto_usuario = mensaje.get("text", "")

        if imagen_cargada:
            # Si viene una lista de archivos o un archivo único
            ruta_img = imagen_cargada[0] if isinstance(imagen_cargada, list) else imagen_cargada
            img = Image.open(ruta_img)
            # Aplicar OCR para extraer el texto del problema de la captura
            texto_extraido = pytesseract.image_to_string(img)
            texto_analisis = (texto_usuario + " " + texto_extraido).lower()
        else:
            texto_analisis = texto_usuario.lower()
    else:
        texto_analisis = str(mensaje).lower()

    # Extraer números del texto (sea escrito o leído por OCR de la imagen)
    numeros = [float(x) for x in re.findall(r"[-+]?\d*\.\d+|\d+", texto_analisis)]

    # 2. ENRUTAMIENTO INTELIGENTE POR MATERIA
    # Termodinámica
    if any(k in texto_analisis for k in ["termodinámica", "van der waals", "compresibilidad", "vapor", "líquido", "presión", "temperatura", "tc", "pc"]):
        T_op = numeros[0] if len(numeros) > 0 else 300.0
        P_bar = numeros[1] if len(numeros) > 1 else 20.0
        P_op = P_bar * 1e5

        Tc, Pc, R = 369.8, 4.25e6, 8.314
        a = (27.0 * (R**2) * (Tc**2)) / (64.0 * Pc)
        b = (R * Tc) / (8.0 * Pc)
        A = (a * P_op) / ((R * T_op)**2)
        B = (b * P_op) / (R * T_op)

        def eq_vdw(z): return z**3 - (1.0 + B)*z**2 + A*z - A*B
        z_v = fsolve(eq_vdw, [0.85])[0]
        z_l = fsolve(eq_vdw, [max(B * 1.1, 0.1)])
        z_l = z_l[0] if isinstance(z_l, np.ndarray) else z_l

        respuesta = f"📚 **[RAG Multimodal / Termodinámica]**\n"
        if texto_extraido: respuesta += f"🔍 *Texto detectado en captura:* `{texto_extraido.strip()[:100]}...`\n\n"
        respuesta += f"⚙️ **Parámetros:** T = `{T_op} K`, P = `{P_bar} bar`\n" \
                     f"- Factor Z Vapor ($Z_v$): `{z_v:.4f}`\n" \
                     f"- Factor Z Líquido ($Z_l$): `{z_l:.4f}`"

    # Balances de Materia y Energía
    elif any(k in texto_analisis for k in ["balance", "energía", "calor", "entalpía", "temperatura", "flujo"]):
        F_molar = numeros[0] if len(numeros) > 0 else 2.0
        Q_aportado = numeros[1] if len(numeros) > 1 else 10000.0
        T_entrada = 25.0

        def balance_riguroso(T_out):
            a, b = 29.0, 0.02
            delta_h = a * (T_out[0] - T_entrada) + (b / 2.0) * (T_out[0]**2 - T_entrada**2)
            return Q_aportado - (F_molar * delta_h)

        T_salida = fsolve(balance_riguroso, [T_entrada + 100.0])[0]
        respuesta = f"📚 **[RAG Multimodal / Balances de Materia y Energía]**\n"
        if texto_extraido: respuesta += f"🔍 *Texto detectado en captura:* `{texto_extraido.strip()[:100]}...`\n\n"
        respuesta += f"⚙️ **Parámetros:** Flujo = `{F_molar} mol/s`, Q = `{Q_aportado} W`\n" \
                     f"- Temperatura de salida: `{T_salida:.2f} °C`"

    # Cinética y Reactores
    elif any(k in texto_analisis for k in ["reactor", "pfr", "tubular", "cinética", "conversión", "reacción", "volumen"]):
        V_reactor = numeros[0] if len(numeros) > 0 else 15.0
        k_val = numeros[1] if len(numeros) > 1 else 0.2303
        v_0, C_A0 = 10.0, 2.0
        F_A0 = v_0 * C_A0

        def modelo_pfr(V, X_arr):
            X = X_arr[0]
            return [0.0] if X >= 1.0 else [(k_val * C_A0 * (1.0 - X)) / F_A0]

        sol = solve_ivp(modelo_pfr, (0.0, V_reactor), [0.0], t_eval=np.linspace(0, V_reactor, 30))
        X_final = sol.y[0][-1]

        respuesta = f"📚 **[RAG Multimodal / Cinética y Reactores]**\n"
        if texto_extraido: respuesta += f"🔍 *Texto detectado en captura:* `{texto_extraido.strip()[:100]}...`\n\n"
        respuesta += f"⚙️ **Parámetros:** Volumen = `{V_reactor} L`, k = `{k_val} min⁻¹`\n" \
                     f"- Conversión final ($X$): `{X_final * 100:.2f}%`"

    # Fenómenos de Transporte
    elif any(k in texto_analisis for k in ["transporte", "conducción", "fourier", "pared", "conductividad", "espesor", "calor"]):
        k_cond = numeros[0] if len(numeros) > 0 else 45.0
        espesor = numeros[1] if len(numeros) > 1 else 5.0
        T1, T2, area = 400.0, 300.0, 2.0
        L = espesor / 100.0
        q_tot = k_cond * area * (T1 - T2) / L

        respuesta = f"📚 **[RAG Multimodal / Fenómenos de Transporte]**\n"
        if texto_extraido: respuesta += f"🔍 *Texto detectado en captura:* `{texto_extraido.strip()[:100]}...`\n\n"
        respuesta += f"⚙️ **Parámetros:** k = `{k_cond} W/m·K`, Espesor = `{espesor} cm`\n" \
                     f"- Flujo de calor total: `{q_tot:.2f} W`"

    # Procesos de Separación
    elif any(k in texto_analisis for k in ["separación", "flash", "rachford", "equilibrio", "vaporización", "fracción", "z_a"]):
        z_A = numeros[0] if len(numeros) > 0 else 0.5
        K_A, K_B = 2.5, 0.5
        z_B = 1.0 - z_A

        def rachford_rice(psi):
            return (z_A * (K_A - 1.0)) / (1.0 + psi[0] * (K_A - 1.0)) + (z_B * (K_B - 1.0)) / (1.0 + psi[0] * (K_B - 1.0))

        psi_val = fsolve(rachford_rice, [0.5])[0]
        x_A = z_A / (1.0 + psi_val * (K_A - 1.0))
        y_A = K_A * x_A

        respuesta = f"📚 **[RAG Multimodal / Procesos de Separación]**\n"
        if texto_extraido: respuesta += f"🔍 *Texto detectado en captura:* `{texto_extraido.strip()[:100]}...`\n\n"
        respuesta += f"⚙️ **Parámetros:** $z_A$ = `{z_A}`\n" \
                     f"- Fracción vaporizada ($\\psi$): `{psi_val:.4f}`"
    else:
        respuesta = f"🤖 **Asistente Multimodal:** He analizado la imagen/texto pero necesito identificar palabras clave de Termodinámica, Balances, Reactores, Transporte o Separación para ejecutar el modelo correcto."

    return respuesta

# Interfaz Gradio con soporte multimodal habilitado (multimodal=True)
demo_multimodal = gr.ChatInterface(
    fn=procesador_multimodal,
    title="💬 Asistente RAG Multimodal - Ingeniería Química (Con Capturas)",
    description="Sube o pega capturas de pantalla de tus problemas de libros o tareas directamente en el chat. El sistema extraerá el texto automáticamente y resolverá el modelo.",
    multimodal=True
)

demo_multimodal.launch(inline=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://04320bd5bce92adf57.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# --- FASE 5: ASISTENTE MULTIMODAL INTELIGENTE CON GEMINI API ---
from google.colab import userdata
import os
import time
import google.genai as genai
import gradio as gr
from PIL import Image

# Cargar la API Key desde los secretos de Google Colab
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Inicializar el cliente de Gemini correctamente
client = genai.Client()

def resolver_problema_ingenieria(mensaje, historial):
    imagen_path = None
    texto_usuario = ""

    # Extraer imagen y texto de la interfaz multimodal de Gradio
    if isinstance(mensaje, dict):
        archivos = mensaje.get("files") or mensaje.get("image")
        texto_usuario = mensaje.get("text", "")
        if archivos:
            imagen_path = archivos[0] if isinstance(archivos, list) else archivos
    else:
        texto_usuario = str(mensaje)

    contents = []

    prompt_sistema = (
        "Eres un profesor experto y doctor en Ingeniería Química, especialista en las 5 materias críticas: "
        "Termodinámica, Balances de Materia y Energía, Cinética y Reactores, Fenómenos de Transporte, y Procesos de Separación. "
        "Analiza detalladamente la imagen o texto provisto por el estudiante. "
        "1. Identifica claramente a qué materia y temática corresponde. "
        "2. Extrae correctamente los datos numéricos reales del problema (ignora números de secciones o incisos aislados). "
        "3. Explica paso a paso el planteamiento teórico y las ecuaciones de los libros de referencia (Bird, Fogler, Smith-Van Ness, etc.) para resolver los incisos planteados. "
        "REGLA DE FORMATO IMPORTANTE: Cuando escribas variables matemáticas, símbolos o unidades, usa formato de texto plano limpio o asegúrate de que sean legibles en párrafos y viñetas, evitando usar bloques complejos de LaTeX puro dentro de listas o viñetas para que la interfaz los muestre correctamente."
    )

    contents.append(prompt_sistema)

    if text_to_send := texto_usuario.strip():
        contents.append(f"Pregunta del estudiante: {text_to_send}")
    else:
        contents.append("Resuelve y explica detalladamente el problema que aparece en esta imagen.")

    if imagen_path:
        img = Image.open(imagen_path)
        contents.append(img)

    # Lógica de reintentos automáticos ante saturación (Error 503)
    max_intentos = 3
    for intento in range(max_intentos):
        try:
            response = client.models.generate_content(
                model='gemini-3.8-flash',
                contents=contents
            )
            return response.text
        except Exception as e:
            error_str = str(e)
            if "503" in error_str or "UNAVAILABLE" in error_str:
                if intento < max_intentos - 1:
                    time.sleep(2) # Espera 2 segundos antes de reintentar
                    continue
            return f"⚠️ Error al procesar con la API de Gemini: {error_str}"

# Interfaz Gradio definitiva con reintento integrado
demo_ia_real = gr.ChatInterface(
    fn=resolver_problema_ingenieria,
    title="🧪 Asistente Experto IA - Ingeniería Química",
    description="Sube la captura de cualquier problema de tus libros (Bird, Fogler, Smith, etc.). La IA leerá la física real del problema y te guiará con el rigor matemático adecuado.",
    multimodal=True
)

demo_ia_real.launch(inline=True, share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4a95c8ac6d9381b808.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [47]:
%%writefile app.py
from google import genai
from google.genai import errors
from PIL import Image
import streamlit as st

# --- CONFIGURACIÓN DE LA INTERFAZ ---
st.set_page_config(
    page_title="Dr. ChemEng - Asistente de Procesos", page_icon="🧪"
)
st.title("🧪 Dr. ChemEng: Tu Asistente de Ingeniería Química")
st.write(
    "Sistema multimodal para el análisis de procesos de separación, balances y"
    " control."
)

# --- PANEL LATERAL PARA LA API KEY Y ARCHIVOS ---
st.sidebar.header("⚙️ Configuración")
user_api_key = st.sidebar.text_input(
    "Ingresa tu Google Gemini API Key:",
    type="password",
    help=(
        "Obtén tu clave gratis en aistudio.google.com. No se guarda en el"
        " código."
    ),
)

st.sidebar.markdown("---")
uploaded_file = st.sidebar.file_uploader(
    "📸 Sube una captura o diagrama (opcional)", type=["png", "jpg", "jpeg"]
)

if uploaded_file is not None:
  image_obj = Image.open(uploaded_file)
  st.sidebar.image(
      image_obj, caption="Imagen cargada", use_container_width=True
  )
else:
  image_obj = None

# --- VALIDACIÓN DE LLAVE ---
if not user_api_key:
  st.warning(
      "⚠️ Por favor, ingresa tu API Key en la barra lateral izquierda para"
      " comenzar a chatear con el asistente."
  )
else:
  # Inicializamos el cliente de Gemini con la llave ingresada
  try:
    client = genai.Client(api_key=user_api_key)
  except Exception as e:
    st.error(f"Error al inicializar el cliente: {e}")

  # Historial de chat
  if "messages" not in st.session_state:
    st.session_state.messages = []

  for message in st.session_state.messages:
    with st.chat_message(message["role"]):
      st.markdown(message["content"])
      if "image" in message and message["image"] is not None:
        st.image(message["image"], width=300)

  # Entrada de texto del usuario
  if user_input := st.chat_input(
      "Escribe tu duda de ingeniería o pide calcular flujos (D, B, etapas)..."
  ):
    # Preparamos el contenido (texto o multimodal)
    if image_obj is not None:
      payload = [user_input, image_obj]
    else:
      payload = user_input

    # Mostramos mensaje del usuario
    st.session_state.messages.append(
        {"role": "user", "content": user_input, "image": image_obj}
    )
    with st.chat_message("user"):
      st.markdown(user_input)
      if image_obj is not None:
        st.image(image_obj, width=300)

    # Llamada a la API de Gemini de forma limpia y directa
    with st.chat_message("assistant"):
      with st.spinner("Analizando problema de ingeniería..."):
        try:
          response = client.models.generate_content(
              model="gemini-3.8-flash", contents=payload
          )
          respuesta = response.text
        except errors.APIError as e:
          respuesta = f"⚠️ Error de API de Google: {str(e)}"
        except Exception as e:
          respuesta = f"⚠️ Error inesperado: {str(e)}"
        st.markdown(respuesta)

    st.session_state.messages.append(
        {"role": "assistant", "content": respuesta, "image": None}
    )

Overwriting app.py


In [48]:
# 1. Instalar Streamlit y Google GenAI
!pip install -q streamlit google-genai pillow

import subprocess
import threading
import time

# Ejecutar Streamlit en segundo plano
def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"])

threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(3)

# 2. Generar el enlace público seguro usando localhost.run (sin cambios)
print("¡Aplicación lista! Copia el enlace web (https://...) que aparece abajo:")
!ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 nokey@localhost.run

¡Aplicación lista! Copia el enlace web (https://...) que aparece abajo:

Welcome to localhost.run!

To set up and manage custom domains go to https://admin.localhost.run/

More details on custom domains (and how to enable subdomains of your custom
domain) at https://localhost.run/docs/custom-domains

If you get a permission denied error check the faq for how to connect with a key or
create a free tunnel without a key at [http://localhost:3000/docs/faq#generating-an-ssh-key].

To explore using localhost.run visit the documentation site:
https://localhost.run/docs/



** your connection id is 34.83.166.94:57500, please mention it if you send me a message about an issue. **

7553576311ee70.lhr.life tunneled with tls termination, https://7553576311ee70.lhr.life
create an account and add your key for a longer lasting domain name. see https://localhost.run/docs/forever-free/ for more information.
Open your tunnel address on your mobile with this QR:

                                         